# Capítulo 6: Obtendo Dados

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 9 de Grus (2019).

> Para escrevê-lo, levei três meses; para concebê-lo, três minutos; para reunir os dados nele, a vida inteira.
>
> — F. Scott Fitzgerald

O [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html) entregou os dados prontos: um `dict` de usuários aqui, uma lista de pares de amizade ali, digitados diretamente no código. Foi proposital — o objetivo daquele capítulo era mostrar o formato do trabalho antes das ferramentas —, mas também foi uma ficção confortável. Na vida real ninguém entrega o dado pronto na sua mão, e o próprio Capítulo 1 avisou que essa lacuna seria cobrada: "obter dado é um problema próprio, e o Capítulo 6 é dedicado a ele."

Este é esse capítulo. Ele não ensina um algoritmo — não há um modelo para treinar nem uma métrica para otimizar. As cinco seções seguem uma escada de fragilidade crescente: primeiro o seu próprio pipe (seção 1), depois o seu próprio disco (seção 2), depois o HTML de outra pessoa (seção 3), depois o contrato formal de outra pessoa — uma API (seção 4) —, e por fim a permissão de outra pessoa para acessá-la (seção 5). Cada degrau depende de menos coisa que você controla, e todo capítulo daqui para frente pressupõe que você sabe descer por ela.

> **🔷 Conceito**
>
> Toda fonte de dado externa — um pipe, um arquivo, uma página, uma API — é um **contrato que você não escreveu e não controla**. O custo de obter o dado é proporcional a quanto controle você perdeu: um arquivo seu (seção 2) você controla inteiramente; a permissão de terceiros para acessar uma rede social (seção 5) você não controla nada. É esse eixo, não a lista de ferramentas, que organiza o capítulo.

O [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html), logo a seguir, parte do princípio de que o dado já está num formato manipulável — listas, `dict`s, colunas — e ensina a limpá-lo e transformá-lo. Este capítulo é o que garante que o dado chegou até ali.

Consequência prática dessa escada: quanto mais baixo o degrau, mais o exemplo depende de um serviço de terceiros continuar no ar e continuar respondendo do mesmo jeito. Por isso os exemplos que precisariam de uma requisição em tempo real aparecem aqui como ilustração, e o que roda de verdade roda sobre dado salvo ou inventado — o serviço muda, a técnica não.

Ao final deste capítulo, você será capaz de:

- Encadear programas Python pela linha de comando usando `sys.stdin` e `sys.stdout`
- Ler e escrever arquivos de texto e arquivos delimitados (CSV, TSV) sem reinventar um parser
- Extrair informação estruturada de uma página HTML com Beautiful Soup, e reconhecer os limites dessa técnica
- Interpretar respostas de API em JSON, e diferenciar uma API autenticada de uma não autenticada
- Explicar por que código que depende de um serviço de terceiros tende a parar de funcionar, e o que fazer diante disso

## Seções

| Seção | Tópico |
|---|---|
| [6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-stdin-e-stdout.html) | stdin e stdout |
| [6.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/02-lendo-arquivos.html) | Lendo Arquivos |
| [6.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/03-raspando-a-web.html) | Raspando a Web |
| [6.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/04-usando-apis.html) | Usando APIs |
| [6.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/05-exemplo-apis-do-twitter.html) | Exemplo: As APIs do Twitter |

## stdin e stdout

> **📌 Nota**
>
> Esta seção corresponde a *stdin and stdout*, do capítulo 9 de Grus (2019).

Se você roda um script Python na linha de comando, dá para conectá-lo a outros programas por meio de um **pipe**: a saída de um vira a entrada do próximo. Python enxerga as duas pontas desse cano por `sys.stdin` (entrada) e `sys.stdout` (saída).

Um exemplo clássico é um `egrep` caseiro — um script que lê linhas de texto e devolve só as que casam com uma expressão regular:

```python
# egrep.py
import sys, re

# sys.argv é a lista de argumentos da linha de comando
# sys.argv[0] é o nome do próprio programa
# sys.argv[1] vai ser a regex passada na linha de comando
regex = sys.argv[1]

# para cada linha recebida pela entrada padrão
for line in sys.stdin:
    # se ela casar com a regex, escreve na saída padrão
    if re.search(regex, line):
        sys.stdout.write(line)
```

E um que conta quantas linhas recebeu:

```python
# line_count.py
import sys

count = 0
for line in sys.stdin:
    count += 1

# print também escreve em sys.stdout
print(count)
```

Encadeados com o caractere `|`, os dois formam um pipeline: `cat arquivo.txt | python egrep.py "[0-9]" | python line_count.py` conta quantas linhas de `arquivo.txt` contêm um dígito, sem que nenhum dos dois scripts precise saber de onde os dados vieram nem para onde vão.

Vale rodar isso de verdade — é a única vez neste capítulo que compensa sair da página, e leva uns trinta segundos. Salve os dois scripts acima como `egrep.py` e `line_count.py` num mesmo diretório, abra um terminal nele, e rode:

```bash
cat egrep.py | python egrep.py 'sys' | python line_count.py
```

Isso conta quantas linhas do próprio `egrep.py` mencionam `sys` — um jeito barato de testar o pipeline sem precisar de nenhum outro arquivo.

O que faz esse encadeamento funcionar é que `sys.stdin` é apenas mais um iterável de linhas, do mesmo jeito que qualquer coisa sobre a qual você já rodou um `for`. É a mesma noção de gerador do [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/05-testes-classes-e-geradores.html) — as linhas chegam sob demanda, uma de cada vez, sem que o programa precise carregar a entrada inteira na memória antes de começar.

Justamente por isso, dá para escrever a mesma lógica como uma função que recebe **qualquer** iterável de linhas — `sys.stdin` é só um caso particular disso:

In [ ]:
import re

def egrep_linhas(linhas, regex):
    """A mesma lógica de egrep.py, mas recebendo o iterável de linhas
    como parâmetro em vez de ler de sys.stdin."""
    for linha in linhas:
        if re.search(regex, linha):
            yield linha

texto = ["banana 42\n", "maçã\n", "uva 7\n"]
list(egrep_linhas(texto, r"[0-9]"))

Rodar `python egrep.py "[0-9]"` na linha de comando é exatamente isto, só que `linhas` é `sys.stdin` em vez de uma lista.

Um terceiro script, um pouco mais elaborado, conta as palavras mais comuns da entrada:

```python
# most_common_words.py
import sys
from collections import Counter

# passe o número de palavras como primeiro argumento
try:
    num_words = int(sys.argv[1])
except (ValueError, IndexError):
    print("uso: most_common_words.py num_palavras")
    sys.exit(1)  # código de saída diferente de zero indica erro

counter = Counter(word.lower()
                  for line in sys.stdin
                  for word in line.strip().split()  # separa por espaço
                  if word)                           # descarta 'palavras' vazias

for word, count in counter.most_common(num_words):
    sys.stdout.write(str(count))
    sys.stdout.write("\t")
    sys.stdout.write(word)
    sys.stdout.write("\n")
```

O `try`/`except` em torno de `int(sys.argv[1])` é o mesmo idioma do [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/02-funcoes-strings-excecoes.html): em vez de checar antes se o argumento é um número válido, tenta converter e trata o erro se a conversão falhar.

> **⚠️ Atenção — Nomeie as exceções que você sabe tratar**
>
> Escrever `except:` pelado nesse `try` sairia mais curto, e seria pior — pela mesma razão que atravessa este capítulo: uma falha silenciosa é pior que uma falha alta.
>
> Rodar o script sem argumento nenhum estoura `IndexError` em `sys.argv[1]`; rodar com um argumento não numérico estoura `ValueError` em `int(...)`. São exatamente essas duas causas que merecem virar "uso incorreto" — e só essas duas. Um `except:` pelado captura isso também, mas captura **qualquer outra coisa** junto: um `NameError` de uma variável digitada errado, um `KeyboardInterrupt`, um `MemoryError`. Cada um desses viraria, em silêncio, a mesma mensagem "uso: most_common_words.py num_palavras" — escondendo exatamente o tipo de erro que você precisaria ver para corrigir. É o que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/02-funcoes-strings-excecoes.html) avisa: "um `except:` pelado engole também os erros que você gostaria de ver."
>
> Nomear as duas exceções específicas entrega o mesmo comportamento visível — a mesma mensagem de uso, para as mesmas duas causas — sem esconder nada além delas.

De novo, a lógica que faz o trabalho de verdade não depende de `sys.stdin` nem de `sys.argv` — só de um iterável de linhas e de um número:

In [ ]:
from collections import Counter

def most_common_words(linhas, num_words):
    counter = Counter(word.lower()
                      for line in linhas
                      for word in line.strip().split()
                      if word)
    return counter.most_common(num_words)

texto2 = ["the quick brown fox\n", "the lazy dog\n", "the fox ran\n"]
most_common_words(texto2, 3)

Rodado de verdade na linha de comando, contra um arquivo de texto grande, a chamada seria:

```bash
cat the_bible.txt | python most_common_words.py 10
```

> **💡 Dica — Na prática: ferramentas de linha de comando**
>
> Se você já usa um terminal Unix, boa parte do que os três scripts acima fazem já existe pronto: `grep` filtra linhas por regex, `wc -l` conta linhas, `sort | uniq -c | sort -rn` conta ocorrências. Vale conhecer essas ferramentas — elas são testadas há décadas e não exigem escrever nada.
>
> Quando o script precisa de mais lógica do que uma linha de shell comporta, a saída não é reinventar `sys.argv` na unha: as bibliotecas `argparse` (na biblioteca padrão) e `click` cuidam de nomear argumentos, gerar mensagens de `--help` e validar tipos, no lugar de indexar `sys.argv` manualmente como os scripts acima fazem. Nenhuma das duas muda a ideia central desta seção — programas que leem de `stdin` e escrevem em `stdout` continuam sendo o jeito de encadear ferramentas pequenas em pipelines maiores.

## Lendo Arquivos

> **📌 Nota**
>
> Esta seção corresponde a *Reading Files*, do capítulo 9 de Grus (2019).

A seção anterior encadeava programas pelo terminal, algo tão efêmero quanto o pipe que os liga. Menos efêmero é o disco: dá para ler e escrever arquivos diretamente no código, e Python torna isso simples.

Os arquivos que aparecem abaixo (`email_addresses.txt`, `tab_delimited_stock_prices.txt` etc.) não existem de antemão: cada um é escrito na hora, num diretório temporário que é apagado no fim da seção.

In [ ]:
import tempfile
from pathlib import Path

pasta = Path(tempfile.mkdtemp(prefix="cap06-"))
pasta

### Fundamentos de arquivos de texto

O primeiro passo para trabalhar com um arquivo de texto é obter um *objeto arquivo* com `open`:

```python
# 'r' significa somente leitura; é o padrão se você não especificar nada
file_for_reading = open('reading_file.txt', 'r')
file_for_reading2 = open('reading_file.txt')

# 'w' é escrita -- destrói o arquivo se ele já existir!
file_for_writing = open('writing_file.txt', 'w')

# 'a' é anexação -- para adicionar ao final do arquivo
file_for_appending = open('appending_file.txt', 'a')

# não esqueça de fechar os arquivos quando terminar
file_for_writing.close()
```

Como é fácil esquecer de fechar um arquivo, o jeito certo é sempre abri-lo dentro de um bloco `with`, que fecha automaticamente ao final:

In [ ]:
with open(pasta / 'exemplo.txt', 'w') as f:
    f.write("primeira linha\nsegunda linha\n")

with open(pasta / 'exemplo.txt') as f:
    conteudo = f.read()

conteudo

`with` é um **gerenciador de contexto** — a mesma garantia de limpeza que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/02-funcoes-strings-excecoes.html) já cobriu para exceções, aplicada a arquivos.

Se você precisa ler um arquivo de texto inteiro, basta iterar sobre as linhas com `for`:

In [ ]:
import re

with open(pasta / 'exemplo2.txt', 'w') as f:
    f.write("# título\nlinha normal\n# outro comentário\nmais uma linha\n")

starts_with_hash = 0
with open(pasta / 'exemplo2.txt') as f:
    for line in f:                    # olha cada linha do arquivo
        if re.match("^#", line):      # usa uma regex para ver se começa com '#'
            starts_with_hash += 1     # se começa, soma 1 à contagem

starts_with_hash

O `re.match` ancora no começo da string — a mesma distinção entre `match` e `search` que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) já cobriu. E iterar assim sobre `f` é o mesmo padrão de gerador da seção anterior: cada `for line in f` produz uma linha por vez, sem carregar o arquivo inteiro na memória de uma vez — o que importa quando o arquivo tem gigabytes.

Toda linha lida desse jeito termina com um caractere de nova linha, então normalmente você vai querer aplicar `.strip()` nela antes de fazer qualquer coisa. Um exemplo: imagine um arquivo com um endereço de e-mail por linha, e você quer montar um histograma dos domínios. A regra para extrair um domínio corretamente é sutil — ela erra em endereços como `joel@m.datasciencester.com` —, mas uma primeira aproximação razoável é pegar tudo depois do `@`:

In [ ]:
from collections import Counter

with open(pasta / 'email_addresses.txt', 'w') as f:
    f.write("joelgrus@gmail.com\n")
    f.write("joel@m.datasciencester.com\n")
    f.write("joelgrus@m.datasciencester.com\n")

def get_domain(email_address: str) -> str:
    """Separa por '@' e devolve o último pedaço"""
    return email_address.lower().split("@")[-1]

# alguns testes
assert get_domain('joelgrus@gmail.com') == 'gmail.com'
assert get_domain('joel@m.datasciencester.com') == 'm.datasciencester.com'

with open(pasta / 'email_addresses.txt', 'r') as f:
    domain_counts = Counter(get_domain(line.strip())
                            for line in f
                            if "@" in line)

domain_counts

### Arquivos delimitados

O arquivo de e-mails tinha um endereço por linha. Mais frequentemente você vai lidar com arquivos que têm vários campos por linha, quase sempre separados por vírgula ou por tabulação — o formato em que o VP de Receita da DataSciencester manda a cotação diária de ações de outro time, por exemplo, sempre que precisa que alguém cruze aquilo com os números da casa.

Em princípio dá para processar esses arquivos lendo linha a linha e cortando pelo separador, mas isso complica rápido quando algum campo contém o próprio caractere separador, ou uma quebra de linha. **Por essa razão, nunca faça o parsing na mão** — use o módulo `csv` da biblioteca padrão (ou o `pandas`, coberto no callout de fechamento desta seção).

Se o arquivo não tem cabeçalho, `csv.reader` itera sobre as linhas já divididas em listas:

In [ ]:
import csv

with open(pasta / 'tab_delimited_stock_prices.txt', 'w') as f:
    f.write("6/20/2014\tAAPL\t90.91\n")
    f.write("6/20/2014\tMSFT\t41.68\n")
    f.write("6/20/2014\tFB\t64.5\n")
    f.write("6/19/2014\tAAPL\t91.86\n")
    f.write("6/19/2014\tMSFT\t41.51\n")
    f.write("6/19/2014\tFB\t64.34\n")

def process(date: str, symbol: str, closing_price: float) -> None:
    # imagine que esta função faz alguma coisa de verdade
    assert closing_price > 0.0

with open(pasta / 'tab_delimited_stock_prices.txt') as f:
    tab_reader = csv.reader(f, delimiter='\t')
    for row in tab_reader:
        date = row[0]
        symbol = row[1]
        closing_price = float(row[2])
        process(date, symbol, closing_price)

"processado sem erro"

É exatamente este `csv.reader`, com a mesma assinatura, que reaparece no [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) lendo `dados/iris.data` linha a linha — o mesmo padrão, só que os campos são medidas de flor em vez de preços de ação.

> **⚠️ Atenção — Linhas malformadas não avisam sozinhas**
>
> `float(row[2])` confia que o terceiro campo é sempre um número. Dados reais quebram essa promessa o tempo todo — um valor ausente, um "N/D", um campo cortado pela metade. Sem tratamento, uma única linha ruim derruba o processamento do arquivo inteiro.
>
> O [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/02-funcoes-strings-excecoes.html) já apresentou o padrão: capture a exceção específica que sabe tratar, não um `except:` pelado. Aplicado aqui, isso significa pular a linha malformada e seguir para a próxima, em vez de estourar:

In [ ]:
with open(pasta / 'precos_com_erro.txt', 'w') as f:
    f.write("6/20/2014\tAAPL\t90.91\n")
    f.write("6/20/2014\tMSFT\tN/D\n")   # preço malformado
    f.write("6/20/2014\tFB\t64.5\n")

precos_validos = []
linhas_puladas = 0

with open(pasta / 'precos_com_erro.txt') as f:
    tab_reader = csv.reader(f, delimiter='\t')
    for row in tab_reader:
        date, symbol, raw_price = row
        try:
            closing_price = float(raw_price)
        except ValueError:
            linhas_puladas += 1
            continue
        precos_validos.append((date, symbol, closing_price))

precos_validos, linhas_puladas

> Duas linhas processadas, uma pulada — e o programa não precisou saber de antemão que a linha do MSFT estava quebrada.

Se o arquivo tem cabeçalho, dá para pular a primeira linha manualmente ou, melhor, usar `csv.DictReader`, que devolve cada linha como um `dict` com o cabeçalho como chaves:

In [ ]:
with open(pasta / 'colon_delimited_stock_prices.txt', 'w') as f:
    f.write("date:symbol:closing_price\n")
    f.write("6/20/2014:AAPL:90.91\n")
    f.write("6/20/2014:MSFT:41.68\n")
    f.write("6/20/2014:FB:64.5\n")

with open(pasta / 'colon_delimited_stock_prices.txt') as f:
    colon_reader = csv.DictReader(f, delimiter=':')
    for dict_row in colon_reader:
        date = dict_row["date"]
        symbol = dict_row["symbol"]
        closing_price = float(dict_row["closing_price"])
        process(date, symbol, closing_price)

"processado sem erro"

Mesmo que o arquivo não tenha cabeçalho, dá para usar `DictReader` passando as chaves como parâmetro `fieldnames`.

Para escrever dados delimitados, use `csv.writer`:

In [ ]:
todays_prices = {'AAPL': 90.91, 'MSFT': 41.68, 'FB': 64.5}

with open(pasta / 'comma_delimited_stock_prices.txt', 'w') as f:
    csv_writer = csv.writer(f, delimiter=',')
    for stock, price in todays_prices.items():
        csv_writer.writerow([stock, price])

(pasta / 'comma_delimited_stock_prices.txt').read_text()

`csv.writer` faz a coisa certa se os campos em si contiverem vírgulas. Um escritor feito à mão provavelmente não faria:

In [ ]:
resultados = [["test1", "success", "Monday"],
              ["test2", "success, kind of", "Tuesday"],
              ["test3", "failure, kind of", "Wednesday"],
              ["test4", "failure, utter", "Thursday"]]

# não faça isto!
with open(pasta / 'bad_csv.txt', 'w') as f:
    for row in resultados:
        f.write(",".join(map(str, row)))  # pode ter vírgulas demais aqui dentro!
        f.write("\n")                     # a linha também pode ter quebras de linha!

print((pasta / 'bad_csv.txt').read_text())

> **⚠️ Atenção**
>
> Repare no que aconteceu: `"success, kind of"` tinha uma vírgula própria, e o arquivo resultante não tem como distinguir essa vírgula das que separam campos. Abrindo `bad_csv.txt` com qualquer leitor de CSV de verdade, a segunda linha vira **quatro** campos em vez de três — e não há mensagem de erro nenhuma avisando disso. É exatamente o tipo de bug silencioso que `csv.writer` existe para evitar.

In [ ]:
import shutil
shutil.rmtree(pasta, ignore_errors=True)

> **💡 Dica — Na prática: `pandas`**
>
> O `pandas` é a biblioteca que a maioria de quem trabalha com dados usa para ler arquivos delimitados:
>
> ```python
> import pandas as pd
>
> precos = pd.read_csv('tab_delimited_stock_prices.txt',
>                      sep='\t', header=None,
>                      names=['date', 'symbol', 'closing_price'])
> ```
>
> Uma linha faz o que o `csv.reader` e o laço `for` acima fizeram juntos, e por padrão o `pandas` já tenta inferir o tipo de cada coluna — `closing_price` já sai como número, sem o `float(row[2])` manual.
>
> O que essa conveniência esconde: a inferência de tipos é uma heurística, não uma garantia. Uma coluna de códigos postais que por acaso são todos numéricos vira `int` silenciosamente, perdendo os zeros à esquerda; uma coluna majoritariamente numérica com um punhado de valores ausentes ou "N/D" pode virar `float` com `NaN` no lugar do texto, sem avisar. O `csv.reader` que você acabou de usar é mais verboso, mas cada conversão de tipo é uma linha que você escreveu e pode inspecionar — o preço da transparência de que fala o restante deste livro.

## Raspando a Web

> **📌 Nota**
>
> Esta seção corresponde a *Scraping the Web*, do capítulo 9 de Grus (2019).

Depois do seu próprio disco vem o HTML de outra pessoa: dá para obter dado raspando páginas web diretamente. Buscar a página é a parte fácil; extrair dela informação estruturada e com sentido, bem menos.

### HTML e como interpretá-lo

Páginas na web são escritas em HTML, no qual o texto é (idealmente) marcado em elementos e seus atributos:

```html
<html>
  <head>
    <title>Uma página web</title>
  </head>
  <body>
    <p id="author">Joel Grus</p>
    <p id="subject">Data Science</p>
  </body>
</html>
```

Num mundo perfeito, onde toda página web fosse marcada de forma semântica em nosso benefício, daria para extrair dado com regras do tipo "ache o elemento `<p>` cujo `id` é `subject` e devolva o texto que ele contém." No mundo real, HTML raramente é bem formado, e quase nunca é anotado — então vamos precisar de ajuda para dar sentido a ele.

Para isso usamos a biblioteca **Beautiful Soup**, que constrói uma árvore a partir dos elementos de uma página e oferece uma interface simples para acessá-los. E usamos a biblioteca **Requests**, um jeito bem mais agradável de fazer requisições HTTP do que qualquer coisa embutida em Python. O parser HTML embutido do Python não é muito tolerante com marcação malformada, então também usamos o parser `html5lib`.

"Tolerante" aqui tem um sentido preciso. Um navegador, ao abrir uma página com uma tag não fechada ou aninhada errado, não trava — ele segue um **algoritmo de recuperação de erros** especificado (o algoritmo de parsing do padrão HTML5), que decide deterministicamente onde cada tag quebrada deveria ter fechado. O parser embutido do Python (`html.parser`) não implementa esse algoritmo, e o problema não é que ele trave: `BeautifulSoup(texto, "html.parser")` sobre uma tag `<p>` aninhada dentro de outra `<p>` não levanta exceção nenhuma — devolve uma árvore **errada**, em silêncio, sem avisar que a marcação estava quebrada. É a mesma classe de falha que a seção anterior ensinou a temer: nenhuma mensagem de erro, só uma resposta incorreta. O `html5lib` implementa o mesmo algoritmo que os navegadores usam, e por isso interpreta a mesma sopa de tags malformadas do jeito que um navegador interpretaria.

Na prática, o HTML viria de uma requisição — `html = requests.get(url).text` — antes de virar `soup = BeautifulSoup(html, 'html5lib')`. Aqui ele vem de uma cópia salva da mesma página, em `dados/getting-data.html`, para que o exemplo dê o mesmo resultado a cada execução. O `BeautifulSoup` recebe exatamente o mesmo texto nos dois casos: só a origem muda.

In [ ]:
from bs4 import BeautifulSoup

with open('dados/getting-data.html') as f:
    html = f.read()

soup = BeautifulSoup(html, 'html5lib')
soup.h1

A partir daqui, tudo funciona com um punhado de métodos simples. Normalmente vamos trabalhar com objetos `Tag`, que correspondem às tags que estruturam a página. Para achar a primeira tag `<p>` (e seu conteúdo):

In [ ]:
first_paragraph = soup.find('p')       # ou simplesmente soup.p

assert str(soup.find('p')) == '<p id="p1">This is the first paragraph.</p>'
first_paragraph

Dá para obter o conteúdo de texto de uma `Tag` pela propriedade `text`:

In [ ]:
first_paragraph_text = soup.p.text
first_paragraph_words = soup.p.text.split()

assert first_paragraph_words == ['This', 'is', 'the', 'first', 'paragraph.']
first_paragraph_words

E dá para extrair os atributos de uma tag tratando-a como um `dict`:

In [ ]:
first_paragraph_id = soup.p['id']         # levanta KeyError se não houver 'id'
first_paragraph_id2 = soup.p.get('id')    # devolve None se não houver 'id'

assert first_paragraph_id == first_paragraph_id2 == 'p1'
first_paragraph_id

Dá para pegar várias tags de uma vez:

In [ ]:
all_paragraphs = soup.find_all('p')             # ou simplesmente soup('p')
paragraphs_with_ids = [p for p in soup('p') if p.get('id')]

assert len(all_paragraphs) == 2
assert len(paragraphs_with_ids) == 1
len(all_paragraphs), len(paragraphs_with_ids)

Frequentemente você vai querer achar tags com uma `class` específica:

In [ ]:
important_paragraphs = soup('p', {'class': 'important'})
important_paragraphs2 = soup('p', 'important')
important_paragraphs3 = [p for p in soup('p')
                         if 'important' in p.get('class', [])]

assert important_paragraphs == important_paragraphs2 == important_paragraphs3
assert len(important_paragraphs) == 1
important_paragraphs

E dá para combinar esses métodos para lógica mais elaborada. Por exemplo, para achar todo elemento `<span>` que está dentro de um `<div>`:

In [ ]:
# Aviso: vai devolver o mesmo <span> várias vezes se ele estiver
# dentro de vários <div>. Seja mais cuidadoso se for o seu caso.
spans_inside_divs = [span
                     for div in soup('div')     # para cada <div> da página
                     for span in div('span')]   # ache cada <span> dentro dele

assert len(spans_inside_divs) == 3
len(spans_inside_divs)

Só esse punhado de recursos já permite fazer bastante coisa. Se você acabar precisando de algo mais elaborado (ou só estiver curioso), vale consultar a [documentação do Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/). O dado que interessa raramente vem rotulado como `class="important"`, claro — é preciso inspecionar o HTML fonte com cuidado, entender a lógica de seleção e prestar atenção a casos de borda para garantir que o dado extraído é o correto.

### Exemplo: de olho no Congresso

O VP de Políticas Públicas da DataSciencester está preocupado com uma possível regulação da indústria de ciência de dados, e pede que você quantifique o que o Congresso norte-americano anda dizendo sobre o tema. Em particular, ele quer achar todos os representantes que têm comunicados de imprensa sobre "data".

Ao tempo em que o Grus escreveu o livro, havia uma página com links para os sites de todos os representantes, em `https://www.house.gov/representatives`, e cada link tinha esta forma:

```html
<td>
  <a href="https://jayapal.house.gov">Jayapal, Pramila</a>
</td>
```

A função abaixo — pura lógica sobre uma string de HTML, sem tocar a rede — verifica se algum `<p>` de um texto menciona uma palavra-chave:

In [ ]:
def paragraph_mentions(text: str, keyword: str) -> bool:
    """
    Devolve True se algum <p> dentro do texto menciona {keyword}
    """
    soup = BeautifulSoup(text, 'html5lib')
    paragraphs = [p.get_text() for p in soup('p')]

    return any(keyword.lower() in paragraph.lower()
               for paragraph in paragraphs)

text = """<body><h1>Facebook</h1><p>Twitter</p></body>"""
assert paragraph_mentions(text, "twitter")       # está dentro de um <p>
assert not paragraph_mentions(text, "facebook")  # não está dentro de um <p>

paragraph_mentions(text, "twitter")

O restante do fluxo — coletar os links da página de representantes, filtrar por regex os que terminam em `.house.gov`, visitar cada site, achar a página de comunicados de imprensa, e procurar "data" em cada um — são dezenas a centenas de requisições HTTP encadeadas, todas dependendo de o `house.gov` estar no ar e responder exatamente como respondia quando o capítulo foi escrito:

```python
from bs4 import BeautifulSoup
import requests
import re
from typing import Dict, Set

url = "https://www.house.gov/representatives"
text = requests.get(url).text
soup = BeautifulSoup(text, "html5lib")

all_urls = [a['href']
           for a in soup('a')
           if a.has_attr('href')]

print(len(all_urls))  # 965 para o Grus, muitos demais

# Precisa começar com http:// ou https://
# Precisa terminar com .house.gov ou .house.gov/
regex = r"^https?://.*\.house\.gov/?$"

# Vamos escrever alguns testes!
assert re.match(regex, "http://joel.house.gov")
assert re.match(regex, "https://joel.house.gov")
assert re.match(regex, "http://joel.house.gov/")
assert re.match(regex, "https://joel.house.gov/")
assert not re.match(regex, "joel.house.gov")
assert not re.match(regex, "http://joel.house.com")
assert not re.match(regex, "https://joel.house.gov/biography")

# E aplica
good_urls = [url for url in all_urls if re.match(regex, url)]

print(len(good_urls))  # ainda 862 para o Grus

# Usa um set para eliminar duplicatas
good_urls = list(set(good_urls))

print(len(good_urls))  # só 431 para o Grus — a Câmara tem 435 cadeiras;
                        # sempre sobra alguma vaga vazia ou site fora do ar

# Na maioria dos sites, há um link para comunicados de imprensa
html = requests.get('https://jayapal.house.gov').text
soup = BeautifulSoup(html, 'html5lib')

# Usa um set porque os links podem aparecer mais de uma vez.
links = {a['href'] for a in soup('a') if 'press releases' in a.text.lower()}
print(links)  # {'/media/press-releases'}

press_releases: Dict[str, Set[str]] = {}

for house_url in good_urls:
    html = requests.get(house_url).text
    soup = BeautifulSoup(html, 'html5lib')
    pr_links = {a['href'] for a in soup('a')
               if 'press releases' in a.text.lower()}
    print(f"{house_url}: {pr_links}")
    press_releases[house_url] = pr_links

for house_url, pr_links in press_releases.items():
    for pr_link in pr_links:
        url = f"{house_url}/{pr_link}"
        text = requests.get(url).text

        if paragraph_mentions(text, 'data'):
            print(f"{house_url}")
            break  # termina com este house_url
```

Este é o exemplo mais datado do capítulo, e vale reparar por quê: os números nos comentários (`965`, `862`, `431`) são o resultado de uma execução específica, não uma constante do site. O `house.gov` pode reformular a página, um representante pode trocar de site, e a lista de representantes muda a cada eleição. O que dura é a técnica — coletar links, filtrar com regex, visitar cada página, aplicar `paragraph_mentions` sobre o resultado —, não os números que ela devolve.

> **📌 Nota**
>
> Raspar um site livremente como o exemplo faz é, em geral, indelicado. A maioria dos sites tem um arquivo `robots.txt` que indica com que frequência (e quais caminhos) você tem permissão de raspar — mas, como observa o próprio Grus, tratando-se do Congresso, não há necessidade de ser particularmente educado.

Vale registrar também uma limitação do exemplo: as páginas de "comunicados de imprensa" de verdade costumam ser paginadas, com só 5 ou 10 comunicados por página. O código acima recupera só os comunicados mais recentes de cada parlamentar — uma solução mais completa precisaria iterar sobre as páginas e recuperar o texto de cada comunicado.

> **💡 Dica — Na prática: além de Beautiful Soup + Requests**
>
> `BeautifulSoup` e `requests` cobrem o caso comum — HTML estático, já presente na resposta HTTP. Duas limitações reais que eles não resolvem:
>
> Primeiro, páginas que montam o conteúdo via JavaScript **depois** de carregadas: `requests.get` devolve o HTML que o servidor mandou, não o que o navegador constrói em cima dele. Para essas, é preciso um navegador de verdade automatizado — ferramentas como **Selenium** ou **Playwright** abrem um navegador (headless ou não), esperam o JavaScript rodar, e só então extraem o HTML resultante.
>
> Segundo, projetos de raspagem maiores — muitas páginas, navegação por links desconhecidos, reexecução periódica — ganham de uma estrutura dedicada como **Scrapy**, que cuida de fila de requisições, novas tentativas, limite de taxa e exportação de dados, em vez de você reimplementar cada uma dessas partes por cima de um `for` com `requests.get`.
>
> Nenhuma das duas muda a lição central: o dado bruto de uma página raspada é HTML mal formado, e alguma coisa — `BeautifulSoup` ou o que quer que você use — vai ter que impor uma estrutura em cima dele antes que a informação vire algo utilizável.

## Usando APIs

> **📌 Nota**
>
> Esta seção corresponde a *Using APIs*, do capítulo 9 de Grus (2019).

Muitos sites e serviços web oferecem **interfaces de programação de aplicações** (APIs), que permitem pedir dado explicitamente, num formato estruturado. Isso poupa o trabalho de raspar — a página pode mudar de layout a qualquer momento, mas uma API bem versionada não muda o formato da resposta sem aviso.

### JSON e XML

Como HTTP é um protocolo para transferir *texto*, o dado que se pede por uma API web precisa ser **serializado** em alguma forma de string. Frequentemente essa serialização usa **JSON** (*JavaScript Object Notation*). Objetos JavaScript se parecem bastante com `dict`s do Python, o que torna sua representação em string fácil de interpretar:

```json
{ "title" : "Data Science Book",
  "author" : "Joel Grus",
  "publicationYear" : 2019,
  "topics" : [ "data", "science", "data science"] }
```

Dá para interpretar JSON usando o módulo `json` do Python. Em particular, vamos usar sua função `loads`, que desserializa uma string representando um objeto JSON, transformando-a num objeto Python:

In [ ]:
import json

serialized = """{ "title" : "Data Science Book",
                  "author" : "Joel Grus",
                  "publicationYear" : 2019,
                  "topics" : [ "data", "science", "data science"] }"""

# interpreta o JSON para criar um dict Python
deserialized = json.loads(serialized)
assert deserialized["publicationYear"] == 2019
assert "data science" in deserialized["topics"]

deserialized

Às vezes um provedor de API só entrega respostas em **XML**:

```xml
<Book>
  <Title>Data Science Book</Title>
  <Author>Joel Grus</Author>
  <PublicationYear>2014</PublicationYear>
  <Topics>
    <Topic>data</Topic>
    <Topic>science</Topic>
    <Topic>data science</Topic>
  </Topics>
</Book>
```

Dá para usar o Beautiful Soup — o mesmo da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/03-raspando-a-web.html) — para extrair dado de XML de forma parecida com o que fizemos com HTML; consulte a documentação para os detalhes.

### Usando uma API sem autenticação

A maioria das APIs hoje em dia exige que você se autentique antes de poder usá-las. Não é capricho dos provedores, mas cobra um bom tanto de código repetitivo que atrapalha a exposição do que interessa. Por isso vamos primeiro olhar a **API do GitHub**, com a qual dá para fazer algumas coisas simples sem autenticação — o bastante, por exemplo, para o VP de Engenharia da DataSciencester descobrir em que dias a equipe mais costuma criar repositórios novos.

A busca em si é uma requisição HTTP:

```python
import requests, json

github_user = "joelgrus"
endpoint = f"https://api.github.com/users/{github_user}/repos"

repos = json.loads(requests.get(endpoint).text)
```

O código acima busca os repositórios da conta `joelgrus` — não a do time de engenharia da DataSciencester do parágrafo anterior. Para apontar a busca a outra conta, basta trocar o valor de `github_user`. O que ele devolve é o estado daquela conta **no momento em que a requisição roda**: rodá-lo hoje e daqui a um mês dá duas listas diferentes.

O que vem depois de `repos` existir é Python puro sobre uma lista de `dict`s — nenhuma linha daqui para baixo toca rede. As datas na resposta chegam como strings, por exemplo `"created_at": "2013-07-05T02:02:28Z"`, e Python não vem com um bom parser de datas embutido; a biblioteca `python-dateutil` cobre esse buraco, e dela você provavelmente só vai precisar da função `dateutil.parser.parse`.

Sobre um `repos` inventado, no mesmo formato que a API devolveria — os cinco repositórios (fictícios) do time de engenharia da DataSciencester —, o processamento fica assim:

In [ ]:
from collections import Counter
from dateutil.parser import parse

repos = [
    {"name": "grafo-de-amizades", "language": "Python",
     "created_at": "2024-03-14T09:12:00Z", "pushed_at": "2024-11-02T17:40:00Z"},
    {"name": "painel-de-metricas", "language": "JavaScript",
     "created_at": "2024-03-21T14:05:00Z", "pushed_at": "2024-10-15T08:22:00Z"},
    {"name": "modelo-de-recomendacao", "language": "Python",
     "created_at": "2024-06-02T11:47:00Z", "pushed_at": "2024-11-20T13:10:00Z"},
    {"name": "relatorios-mensais", "language": "R",
     "created_at": "2024-07-19T16:30:00Z", "pushed_at": "2024-09-05T10:00:00Z"},
    {"name": "api-interna", "language": "Python",
     "created_at": "2024-09-08T08:55:00Z", "pushed_at": "2024-11-25T19:03:00Z"},
]

# Em que meses e dias da semana a equipe mais costuma criar repositórios?
dates = [parse(repo["created_at"]) for repo in repos]
month_counts = Counter(date.month for date in dates)
weekday_counts = Counter(date.weekday() for date in dates)  # segunda=0 ... domingo=6

# E em que linguagem estão os repositórios modificados mais recentemente?
last_5_repositories = sorted(repos,
                             key=lambda r: r["pushed_at"],
                             reverse=True)[:5]

last_5_languages = [repo["language"] for repo in last_5_repositories]

month_counts, weekday_counts, last_5_languages

Contra a API de verdade, `repos` teria dezenas ou centenas de entradas em vez de cinco, mas a lógica que extrai mês, dia da semana e linguagem é exatamente esta — rodando sobre um `dict` Python comum, o mesmo tipo de dado que você manipulou o livro inteiro, só que ele teria chegado pela rede em vez de escrito à mão aqui.

O formato em que o dado chega já é uma decisão que ecoa adiante: JSON quebrado em campos como `language` e `created_at` deixa pronto exatamente o tipo de [atributo](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html) que um modelo consegue usar — uma API que devolvesse só um parágrafo de texto livre sobre cada repositório teria deixado bem menos pronto.

Normalmente, você não vai trabalhar com APIs neste nível baixo de "montar a requisição e interpretar a resposta na mão". Um dos benefícios de usar Python é que alguém provavelmente já escreveu uma biblioteca para praticamente qualquer API que interesse. Quando essas bibliotecas são bem feitas, elas poupam bastante trabalho de descobrir os detalhes mais espinhosos do acesso à API. (Quando não são bem feitas, ou quando ficam desatualizadas em relação à versão real da API que envolvem, elas podem causar dores de cabeça enormes.)

Mesmo assim, eventualmente você vai precisar escrever sua própria biblioteca de acesso a alguma API (ou, mais provavelmente, depurar por que a de outra pessoa não está funcionando) — então vale conhecer alguns desses detalhes.

### Encontrando APIs

Para uma API específica, vale checar se já existe uma biblioteca Python pronta antes de montar as chamadas na mão: sites como Yelp, Instagram e Spotify têm as suas próprias, e o Grus recomendava uma lista de *wrappers* que o [Real Python](https://realpython.com/) mantinha no GitHub — o tipo de link específico que data primeiro num livro, como a seção seguinte está prestes a ilustrar. Quando não existe nenhuma pronta, sempre resta a raspagem — o último recurso do cientista de dados, coberto na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/03-raspando-a-web.html).

> **💡 Dica — Na prática: bibliotecas de acesso a APIs**
>
> O padrão `requests.get` + `json.loads` que você acabou de escrever é exatamente o que uma biblioteca de acesso a API encapsula. `PyGithub`, por exemplo, faz para a API do GitHub o que você faria à mão acima, só que devolvendo objetos Python tipados (`repo.created_at` já vem como `datetime`, não como string) em vez de `dict`s crus:
>
> ```python
> from github import Github
>
> g = Github()
> repos = g.get_user("joelgrus").get_repos()
> ```
>
> O que essas bibliotecas escondem é justamente a parte chata: paginação (a maioria das APIs devolve resultados em páginas de 20 ou 100 itens, e alguém precisa pedir a próxima), limite de taxa (a maioria impõe um número máximo de requisições por hora, e passar do limite derruba sua aplicação até o limite resetar) e nova tentativa em caso de falha de rede transitória. Nenhuma dessas preocupações aparece nos exemplos deste capítulo — eles fazem uma chamada só —, mas qualquer coisa que colete dado de uma API em produção precisa lidar com as três. A [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/05-exemplo-apis-do-twitter.html) mostra outra peça desse quebra-cabeça: autenticação.

## Exemplo: As APIs do Twitter

> **📌 Nota**
>
> Esta seção corresponde a *Example: Using the Twitter APIs*, do capítulo 9 de Grus (2019).

Redes sociais como o Twitter são uma fonte valiosa de dado: notícias em tempo real, reação a eventos correntes, links sobre um assunto específico. E, como boa parte do dado interessante do mundo, o acesso a ele é controlado por uma API.

O acesso gratuito a essa API foi descontinuado depois da publicação do livro-texto, e a biblioteca `twython`, usada aqui, ficou sem manutenção: o código a seguir não funciona mais como escrito, e vale pelo fluxo que ele mostra, não pelo serviço que ele acessa.

### Obtendo credenciais

Para usar a API de uma rede social, tipicamente é preciso criar uma conta de desenvolvedor, abrir um aplicativo dentro dela, e receber um par de chaves — uma **chave de consumidor** (*consumer key*) e um **segredo de consumidor** (*consumer secret*), às vezes chamados de *API key* e *API secret key*. Essas chaves identificam a **aplicação**, não o usuário final — é o aplicativo que está pedindo acesso à API, agindo em nome de alguém.

> **⚠️ Atenção — Chave não é coisa para deixar no código**
>
> As chaves de aplicação são, na prática, uma senha: quem as tiver pode fazer chamadas em nome do seu aplicativo, até o limite que a API permitir. Duas práticas comuns para não deixá-las expostas:
>
> - Guardá-las num arquivo separado (por exemplo `credentials.json`) que **não é** versionado no controle de código, lendo-o com `json.loads` quando o programa inicia.
> - Guardá-las em variáveis de ambiente e lê-las com `os.environ.get`, o padrão que o próprio código do Grus usa abaixo.
>
> Nunca as escreva direto no código-fonte, e muito menos em um repositório público — mesmo privado, um commit antigo com a chave continua lá no histórico depois de "removida" de um commit novo.

### Usando o Twython

A parte mais delicada de usar a API do Twitter, escreve o Grus, é a autenticação — e isso vale para a maioria das APIs. Provedores de API querem garantir que você está autorizado a acessar os dados e que não está estourando os limites de uso deles; também querem saber quem está acessando.

Existem, tipicamente, dois níveis de autenticação: um mais simples (**OAuth 2**), que basta para operações de leitura como buscas; e um mais elaborado (**OAuth 1**), necessário para ações que agem em nome de um usuário — postar, ou (o caso que interessa aqui) conectar-se ao fluxo contínuo de dados.

Com as chaves em mãos, a biblioteca `Twython` automatizava boa parte do fluxo OAuth 1:

```python
import os

# Sinta-se à vontade para colocar sua chave e segredo diretamente
CONSUMER_KEY = os.environ.get("TWITTER_CONSUMER_KEY")
CONSUMER_SECRET = os.environ.get("TWITTER_CONSUMER_SECRET")

import webbrowser
from twython import Twython

# Obtém um cliente temporário para buscar uma URL de autenticação
temp_client = Twython(CONSUMER_KEY, CONSUMER_SECRET)
temp_creds = temp_client.get_authentication_tokens()
url = temp_creds['auth_url']

# Agora visite essa URL para autorizar a aplicação e obter um PIN
print(f"visite {url} e pegue o código PIN, colando-o abaixo")
webbrowser.open(url)
PIN_CODE = input("digite o código PIN: ")

# Agora usamos o PIN_CODE para obter os tokens de verdade
auth_client = Twython(CONSUMER_KEY,
                      CONSUMER_SECRET,
                      temp_creds['oauth_token'],
                      temp_creds['oauth_token_secret'])
final_step = auth_client.get_authorized_tokens(PIN_CODE)
ACCESS_TOKEN = final_step['oauth_token']
ACCESS_TOKEN_SECRET = final_step['oauth_token_secret']

# E obtemos uma nova instância de Twython usando-os.
twitter = Twython(CONSUMER_KEY, CONSUMER_SECRET, ACCESS_TOKEN, ACCESS_TOKEN_SECRET)
```

Vale reconhecer a forma desse fluxo, mesmo sem a `twython`: um cliente **temporário** troca a chave da aplicação por uma URL; um humano visita essa URL, autoriza a aplicação de dentro da própria conta, e recebe um PIN de volta; esse PIN é trocado por um **token de acesso** de verdade, que passa a autenticar as chamadas seguintes. É o mesmo desenho geral do OAuth que aparece atrás de "Entrar com o Google" ou "Entrar com o GitHub" em incontáveis sites — a aplicação nunca vê a senha do usuário, só um token que ele concedeu explicitamente e pode revogar depois.

Autenticado, o próximo passo seria fazer buscas:

```python
# Busca tweets contendo a frase "data science"
for status in twitter.search(q='"data science"')["statuses"]:
    user = status["user"]["screen_name"]
    text = status["text"]
    print(f"{user}: {text}\n")
```

### Busca contra fluxo contínuo

A API de busca devolve só um punhado de resultados recentes — útil para uma consulta pontual, pouco útil quando o que se quer é **muito** dado. Para isso existia a **API de streaming**, que conectava a uma amostra do fluxo contínuo de mensagens públicas do Twitter, exigindo o nível mais elaborado de autenticação (OAuth 1, o mesmo obtido acima).

Para acessá-la com `Twython`, era preciso definir uma classe que herdasse de `TwythonStreamer` e sobrescrevesse o método `on_success` — chamado toda vez que um novo dado chegava — e, possivelmente, `on_error`:

```python
from twython import TwythonStreamer

# Acumular dado numa variável global é uma prática pobre,
# mas torna o exemplo bem mais simples
tweets = []

class MyStreamer(TwythonStreamer):
    def on_success(self, data):
        """
        O que fazer quando o Twitter nos manda dado?
        Aqui, data será um dict do Python representando um tweet.
        """
        # Só queremos coletar tweets em inglês
        if data.get('lang') == 'en':
            tweets.append(data)
            print(f"tweet recebido #{len(tweets)}")

        # Para quando já tivermos coletado o suficiente
        if len(tweets) >= 100:
            self.disconnect()

    def on_error(self, status_code, data):
        print(status_code, data)
        self.disconnect()

stream = MyStreamer(CONSUMER_KEY, CONSUMER_SECRET,
                    ACCESS_TOKEN, ACCESS_TOKEN_SECRET)

# começa a consumir mensagens públicas que contenham a palavra-chave 'data'
stream.statuses.filter(track='data')

# se em vez disso quiséssemos consumir uma amostra de *todas* as mensagens públicas
# stream.statuses.sample()
```

`MyStreamer` se conectaria ao fluxo do Twitter e esperaria por dados. A cada mensagem recebida (aqui, um tweet representado como um objeto Python), o Twitter chamaria `on_success`, que a acumularia na lista `tweets` se o idioma fosse inglês, e desconectaria depois de coletar 100.

A partir daí, seria possível analisar o resultado — por exemplo, achando as hashtags mais comuns:

```python
from collections import Counter

top_hashtags = Counter(hashtag['text'].lower()
                       for tweet in tweets
                       for hashtag in tweet["entities"]["hashtags"])

print(top_hashtags.most_common(5))
```

> **📌 Nota**
>
> Cada tweet carrega bastante dado estruturado além do texto — o idioma, as hashtags, menções, geolocalização quando disponível. Num projeto de verdade, você não ia querer depender de uma `list` em memória para guardar os tweets coletados; ia querer salvá-los num arquivo ou banco de dados, para tê-los de forma permanente — a `list tweets` acima desaparece assim que o processo termina.

> **💡 Dica — Na prática: o que sobrevive desta seção**
>
> **Uma dependência externa que morre é o destino normal de todo código que depende de terceiros** — não uma falha de projeto do Grus: o acesso gratuito à API do Twitter que o livro usa foi descontinuado depois da publicação, o produto trocou de nome, e a própria `twython` está sem manutenção.
>
> O que sobrevive não é "copie isto e funcione" — é entender **autenticação de API** como problema geral. Toda API atual que exige login segue alguma variação do mesmo roteiro:
>
> - **Uma chave de aplicação**, obtida uma vez, num painel de desenvolvedor.
> - **Um fluxo de autorização** — hoje quase sempre OAuth 2 com algum tipo de token de curta duração e um *refresh token* para renová-lo, em vez do PIN manual do OAuth 1 usado aqui.
> - **Um limite de taxa** (*rate limit*), que toda aplicação séria respeita e cujo estouro é a causa mais comum de "minha integração parou de funcionar do nada".
> - **Uma distinção entre busca pontual e fluxo contínuo** — a maioria das APIs modernas de redes sociais, quando ainda oferece acesso de terceiros, mantém essa mesma divisão entre "peça um punhado de resultados agora" e "assine um canal e receba dado conforme ele acontece".
>
> O que este livro não pode fazer é lhe dar um exemplo que continue funcionando: qualquer nome de API específico citado aqui corre o risco de ficar desatualizado antes mesmo deste material ser publicado. O que fica é saber reconhecer essas quatro peças — chave, fluxo de autorização, limite de taxa, busca versus fluxo — na documentação de qualquer API nova que você precisar aprender. Os nomes específicos deste capítulo (`Twython`, `CONSUMER_KEY`) morreram junto com o produto que descreviam; a forma do problema, não.

### Adiante

O que você tem, ao final deste capítulo, não é mais um pipe, um arquivo, uma árvore de tags ou uma resposta de API — é o mesmo material com que o livro trabalha desde o Capítulo 1: listas e `dict`s do Python, na memória. É para lá que qualquer uma das cinco seções converge, não importa quão frágil ou robusta tenha sido a fonte.

O [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html) assume que esse trabalho já foi feito, e começa exatamente onde este termina: explorando, limpando e transformando o dado que agora está em suas mãos.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 9 de Grus (2019) sugere:

- [pandas](https://pandas.pydata.org/) é a biblioteca principal que quem trabalha com dados usa para lidar com — e, em particular, importar — dado. Este livro não a usa na implementação de nada, pelo mesmo motivo de sempre: o objetivo aqui é ver o mecanismo por baixo, não a API de alto nível que o esconde. Mas fora deste livro, `pandas.read_csv` e `pandas.read_json` substituem boa parte do código escrito à mão neste capítulo.
- [Scrapy](https://scrapy.org/) é uma biblioteca completa para construir raspadores web complicados, que fazem coisas como seguir links desconhecidos automaticamente — muito além do que `requests` + `BeautifulSoup` fazem na [seção 6.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/03-raspando-a-web.html).
- [Kaggle](https://www.kaggle.com/) hospeda uma grande coleção de conjuntos de dados, junto com competições organizadas em torno deles — um jeito de praticar sobre dado real sem primeiro ter que resolver o problema deste capítulo.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.